# 1.1 Reddit — Exploratory Data Analysis

**Pipeline:**  
Bronze → language filter (English only) → text cleaning → buzz labels → **Silver** (`reddit_posts_clean.parquet` · `reddit_comments_clean.parquet`)

Downstream notebooks: network (1.2) · textual (1.3) · NLP (1.4) · sentiment (1.4)

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.join(os.path.abspath("."), "..", ".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from house_style import *
from Functions.text_preprocessing import apply_text_cleaning
from Functions.buzz_column import add_buzz_labels

apply_style()

BRONZE_DIR = Path("../../Data/1_Bronze/Reddit")
SILVER_DIR = Path("../../Data/2_Silver/Reddit")

POST_COLS    = ["id", "author", "created_utc", "subreddit", "title", "selftext",
                "score", "upvote_ratio", "num_comments", "permalink"]
COMMENT_COLS = ["id", "author", "created_utc", "subreddit", "body", "score", "permalink"]

posts    = pd.read_parquet(BRONZE_DIR / "reddit_posts_filtered.parquet",    columns=POST_COLS)
comments = pd.read_parquet(BRONZE_DIR / "reddit_comments_filtered.parquet", columns=COMMENT_COLS)

posts["created_utc"]    = pd.to_datetime(posts["created_utc"],    utc=True)
comments["created_utc"] = pd.to_datetime(comments["created_utc"], utc=True)

print(f"Posts    : {len(posts):,} rows x {posts.shape[1]} columns")
print(f"Comments : {len(comments):,} rows x {comments.shape[1]} columns")

## 1. Bronze — raw data structure

In [ ]:
posts.head(3)

In [ ]:
comments.head(3)

In [ ]:
print("--- POSTS ---")
posts.info()
print()
print("--- COMMENTS ---")
comments.info()

## 2. Text preprocessing → Buzz labels → Silver

Posts use `title + selftext` as the text field. Comments use `body`.  
Deleted/removed content (`[deleted]`, `[removed]`) is treated as empty.

In [ ]:
# Posts: combine title + selftext into a single text column
def _combine_post_text(row):
    title    = str(row["title"])    if pd.notna(row["title"])    else ""
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    selftext = "" if selftext in ("[deleted]", "[removed]") else selftext
    return (title + " " + selftext).strip()

posts["text"]    = posts.apply(_combine_post_text, axis=1)

# Comments: body is the text column
comments["text"] = comments["body"].replace({"[deleted]": "", "[removed]": ""})

# Apply cleaning + English filter
posts_clean    = apply_text_cleaning(posts,    text_col="text", filter_english=True)
comments_clean = apply_text_cleaning(comments, text_col="text", filter_english=True)

# Add buzz labels
posts_clean    = add_buzz_labels(posts_clean,    text_col="text_clean")
comments_clean = add_buzz_labels(comments_clean, text_col="text_clean")

# Save to Silver
SILVER_DIR.mkdir(parents=True, exist_ok=True)
posts_clean.to_parquet(   SILVER_DIR / "reddit_posts_clean.parquet",    index=False)
comments_clean.to_parquet(SILVER_DIR / "reddit_comments_clean.parquet", index=False)

print(f"Saved posts    : {len(posts_clean):,} rows x {posts_clean.shape[1]} cols")
print(f"Saved comments : {len(comments_clean):,} rows x {comments_clean.shape[1]} cols")

## 3. Exploratory Analysis

All analyses below run on the Silver data: English only, cleaned, labelled.

In [ ]:
posts_clean[["title", "text_clean", "word_count", "candidate", "subreddit"]].head(5)

### Key counts per candidate

In [ ]:
COLORS = {"TrumpBuzz": REPUBLICAN, "HarrisBuzz": DEMOCRAT, "ElectionBuzz": NEUTRAL}

summary = pd.DataFrame({
    "posts"           : posts_clean.groupby("candidate").size(),
    "unique_authors"  : posts_clean.groupby("candidate")["author"].nunique(),
    "avg_score"       : posts_clean.groupby("candidate")["score"].mean().round(1),
    "avg_comments"    : posts_clean.groupby("candidate")["num_comments"].mean().round(1),
    "avg_upvote_ratio": posts_clean.groupby("candidate")["upvote_ratio"].mean().round(3),
})
summary

### Posts by subreddit

In [ ]:
sub_counts = posts_clean["subreddit"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(sub_counts.index[::-1], sub_counts.values[::-1], color=NEUTRAL)
ax.set_xlabel("Number of posts", color=TEXT_MUTED)
ax.set_title("Posts per subreddit (Silver)", color=TEXT_PRIMARY, pad=10)
plt.tight_layout()
plt.show()

### Daily post volume

In [ ]:
posts_clean["date"] = posts_clean["created_utc"].dt.normalize()

daily = posts_clean.groupby(["date", "candidate"]).size().reset_index(name="posts")

fig, ax = plt.subplots(figsize=(13, 4))
for cand, grp in daily.groupby("candidate"):
    ax.plot(grp["date"], grp["posts"], label=cand,
            color=COLORS.get(cand, NEUTRAL), linewidth=1.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_ylabel("Posts per day", color=TEXT_MUTED)
ax.set_title("Daily Reddit post volume by candidate cluster", color=TEXT_PRIMARY, pad=10)
ax.legend(frameon=False, labelcolor=TEXT_PRIMARY)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Score distribution (log scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for cand, grp in posts_clean.groupby("candidate"):
    ax.hist(np.log1p(grp["score"].clip(lower=0)), bins=40, alpha=0.55,
            label=cand, color=COLORS.get(cand, NEUTRAL))
ax.set_xlabel("log(1 + score)", color=TEXT_MUTED)
ax.set_ylabel("Count", color=TEXT_MUTED)
ax.set_title("Score distribution by candidate cluster", color=TEXT_PRIMARY, pad=10)
ax.legend(frameon=False, labelcolor=TEXT_PRIMARY)
plt.tight_layout()
plt.show()

### Word count distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(posts_clean["word_count"], bins=40, color=NEUTRAL, edgecolor="none")
ax.set_xlabel("Words per post (after cleaning)", color=TEXT_MUTED)
ax.set_ylabel("Count", color=TEXT_MUTED)
ax.set_title("Word count — posts", color=TEXT_PRIMARY, pad=10)

ax = axes[1]
ax.hist(comments_clean["word_count"].clip(upper=200), bins=40, color=NEUTRAL, edgecolor="none")
ax.set_xlabel("Words per comment (after cleaning)", color=TEXT_MUTED)
ax.set_ylabel("Count", color=TEXT_MUTED)
ax.set_title("Word count — comments", color=TEXT_PRIMARY, pad=10)

plt.tight_layout()
plt.show()

empty_posts    = (posts_clean["word_count"]    == 0).sum()
empty_comments = (comments_clean["word_count"] == 0).sum()
print(f"Empty posts    after cleaning : {empty_posts:,} ({empty_posts/len(posts_clean)*100:.1f}%)")
print(f"Empty comments after cleaning : {empty_comments:,} ({empty_comments/len(comments_clean)*100:.1f}%)")